# Fashion Sustainability Cluster Analysis - Full Pipeline

=============================================================
 Fashion Sustainability Cluster Analysis — FULL PIPELINE
=============================================================
 Purpose: Complete K-means + hierarchical clustering analysis
          producing every figure and table used in the
          dissertation (Chapter 4), including the primary K=2
          solution and the supplementary K=3/K=4 analyses.

 Input:   Fashion_Sustainability_Final_Dataset.csv
          (produced by script_02_merge_sources.py +
           script_03_add_performance_data.py)

 Outputs (all saved to /outputs/):
   fig1_elbow_silhouette.png       - Elbow + silhouette (K=2-8)
   fig2_pca_cluster_map.png        - PCA scatter, K=2 primary
   fig3_radar_charts.png           - Radar charts, K=2 clusters
   fig4_greenwashing_matrix.png    - Disclosure vs CDP matrix
   fig5_dendrogram.png             - Hierarchical dendrogram
   fig6_subsegment_region.png      - Sub-segment/region heatmaps
   fig7_silhouette_plot.png        - Per-company silhouette plot
   fig8_k3_k4_pca.png              - Supplementary K=3/K=4 PCA
   fig9_cluster_comparison_table.png - K=2/3/4 comparison table
   fig10_k3_k4_radars.png          - Supplementary K=3/K=4 radars
   Fashion_Clustered_Dataset.csv   - Final dataset w/ cluster labels

 Usage:
   pip install pandas scikit-learn scipy matplotlib seaborn numpy openpyxl
   python script_04_full_clustering_analysis.py
=============================================================


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

## CONFIG


In [ ]:
INPUT_CSV = 'Fashion_Sustainability_Final_Dataset.csv'
OUT = 'outputs/'
import os
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 200,
    'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
})

CLUSTER_COLORS_K2 = ['#1A3C5E', '#2E9E6E']
CLUSTER_COLORS_K3 = ['#1A3C5E', '#E67E22', '#2E9E6E']
CLUSTER_COLORS_K4 = ['#1A3C5E', '#8E44AD', '#E67E22', '#2E9E6E']

## 1. LOAD DATA


In [ ]:
print("=" * 60)
print("FASHION SUSTAINABILITY CLUSTER ANALYSIS — FULL PIPELINE")
print("=" * 60)

df = pd.read_csv(INPUT_CSV)
print(f"\nDataset loaded: {df.shape[0]} companies, {df.shape[1]} variables")

# CDP: Not Disclosed -> 0 (non-disclosure is informative, not missing at random)
df['CDP_Score_Encoded'] = df['CDP_Score_Numeric'].fillna(0)
print(f"CDP Not Disclosed companies encoded as 0: {df['CDP_Score_Numeric'].isna().sum()}")

CLUSTER_VARS = [
    'Carbon_Emissions_Score_pct', 'Energy_Renewables_Score_pct',
    'Supply_Chain_Score_pct', 'Social_Workers_Score_pct',
    'Governance_Strategy_Score_pct', 'Materials_Products_Score_pct',
    'CDP_Score_Encoded', 'Has_Net_Zero_Target'
]
THEME_VARS = CLUSTER_VARS[:6]
THEME_LABELS = ['Carbon &\nEmissions', 'Energy &\nRenewables', 'Supply\nChain',
                'Social &\nWorkers', 'Governance', 'Materials']

X = df[CLUSTER_VARS].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"Clustering variables: {len(CLUSTER_VARS)} | Missing values: {X.isna().sum().sum()}")

## 2. ELBOW METHOD + SILHOUETTE (K=2 to K=8)


In [ ]:
print("\n--- STEP 1: Elbow Method & Silhouette Analysis (K=2 to K=8) ---")
inertias, sil_scores = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_)
    sil_scores.append(sil)
    print(f"  K={k}: Inertia={km.inertia_:.1f}, Silhouette={sil:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(list(K_range), inertias, 'o-', color='#1A3C5E', linewidth=2.5, markersize=8)
ax1.set_xlabel('Number of Clusters (K)'); ax1.set_ylabel('Within-Cluster Sum of Squares')
ax1.set_title('Elbow Method — Optimal K Selection'); ax1.set_xticks(list(K_range)); ax1.grid(True, alpha=0.3)
for k, v in zip(K_range, inertias):
    ax1.annotate(f'{v:.0f}', (k, v), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9)

ax2.plot(list(K_range), sil_scores, 'o-', color='#2E9E6E', linewidth=2.5, markersize=8)
ax2.axhline(y=0.3, color='red', linestyle='--', alpha=0.7, label='Min threshold (0.3)')
ax2.set_xlabel('Number of Clusters (K)'); ax2.set_ylabel('Average Silhouette Score')
ax2.set_title('Silhouette Analysis — Cluster Quality'); ax2.set_xticks(list(K_range)); ax2.legend(); ax2.grid(True, alpha=0.3)
for k, s in zip(K_range, sil_scores):
    ax2.annotate(f'{s:.3f}', (k, s), textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9)

plt.suptitle('K-Means Optimisation: Fashion Sustainability Clustering', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(OUT + 'fig1_elbow_silhouette.png'); plt.close()
print("  Saved: fig1_elbow_silhouette.png")

best_k = list(K_range)[sil_scores.index(max(sil_scores))]
print(f"\nOptimal K (highest silhouette): K={best_k}, score={max(sil_scores):.4f}")

## 3. PRIMARY K=2 SOLUTION


In [ ]:
print(f"\n--- STEP 2: Primary Solution K={best_k} ---")
OPTIMAL_K = best_k
km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=20, max_iter=500)
km_final.fit(X_scaled)
df['Cluster'] = km_final.labels_
final_sil = silhouette_score(X_scaled, km_final.labels_)
print(f"Silhouette: {final_sil:.4f} | Sizes: {pd.Series(km_final.labels_).value_counts().sort_index().to_dict()}")

# Label clusters by descending overall disclosure score
cluster_overall = df.groupby('Cluster')['Overall_Disclosure_Score_pct'].mean().sort_values(ascending=False)
rank_order = cluster_overall.index.tolist()
label_names = ['Cluster 1 — Leaders', 'Cluster 2 — Developing']
rank_to_label = {c: label_names[i] for i, c in enumerate(rank_order)}
df['Cluster_Label'] = df['Cluster'].map(rank_to_label)

cluster_means = df.groupby('Cluster_Label')[CLUSTER_VARS].mean()
print("\nCluster mean scores:")
print(cluster_means.round(2).to_string())

## 4. PCA VISUALISATION (K=2)


In [ ]:
print("\n--- STEP 3: PCA Visualisation ---")
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'], df['PCA2'] = X_pca[:, 0], X_pca[:, 1]
var_exp = pca.explained_variance_ratio_
print(f"Variance explained: PC1={var_exp[0]:.1%}, PC2={var_exp[1]:.1%}")

fig, ax = plt.subplots(figsize=(13, 9))
labels_sorted = sorted(df['Cluster_Label'].unique())
for i, label in enumerate(labels_sorted):
    mask = df['Cluster_Label'] == label
    ax.scatter(df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'], c=CLUSTER_COLORS_K2[i],
               label=label, s=100, alpha=0.85, edgecolors='white', linewidth=0.8, zorder=3)
for _, row in df.iterrows():
    ax.annotate(row['Company'].split(' ')[0], (row['PCA1'], row['PCA2']), fontsize=6.5,
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', alpha=0.75)
ax.set_xlabel(f'PC1 ({var_exp[0]:.1%} variance)'); ax.set_ylabel(f'PC2 ({var_exp[1]:.1%} variance)')
ax.set_title('Sustainability Maturity Clusters — PCA Visualisation\n64 Global Fashion Companies',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left'); ax.grid(True, alpha=0.25)
ax.axhline(y=0, color='grey', linewidth=0.5, alpha=0.4); ax.axvline(x=0, color='grey', linewidth=0.5, alpha=0.4)
ax.text(0.02, 0.02, f'Average Silhouette Score: {final_sil:.3f} | K={OPTIMAL_K}', transform=ax.transAxes,
        fontsize=9, bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8))
plt.tight_layout(); plt.savefig(OUT + 'fig2_pca_cluster_map.png'); plt.close()
print("  Saved: fig2_pca_cluster_map.png")

## 5. RADAR CHARTS (K=2)


In [ ]:
print("\n--- STEP 4: Radar Charts (K=2) ---")
N = len(THEME_VARS)
angles = [n / float(N) * 2 * np.pi for n in range(N)]; angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(14, 7), subplot_kw=dict(polar=True))
fig.suptitle('Cluster Sustainability Profiles — Disclosure Theme Scores (%)', fontsize=14, fontweight='bold', y=1.02)
overall_data = df[THEME_VARS].mean().values.tolist(); overall_data += overall_data[:1]
for idx, label in enumerate(labels_sorted):
    ax = axes[idx]
    vals = df[df['Cluster_Label'] == label][THEME_VARS].mean().values.tolist(); vals += vals[:1]
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(THEME_LABELS, size=10)
    ax.set_ylim(0, 100); ax.set_yticks([20, 40, 60, 80, 100])
    ax.plot(angles, overall_data, 'o--', color='grey', linewidth=1.5, alpha=0.6, label='Industry avg')
    ax.fill(angles, overall_data, alpha=0.08, color='grey')
    ax.plot(angles, vals, 'o-', color=CLUSTER_COLORS_K2[idx], linewidth=2.5, label=label.split('—')[1].strip())
    ax.fill(angles, vals, alpha=0.25, color=CLUSTER_COLORS_K2[idx])
    n_c = (df['Cluster_Label'] == label).sum()
    ax.set_title(f"{label}\nn={n_c} companies", size=11, fontweight='bold', pad=15)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout(); plt.savefig(OUT + 'fig3_radar_charts.png'); plt.close()
print("  Saved: fig3_radar_charts.png")

## 6. GREENWASHING MATRIX


In [ ]:
print("\n--- STEP 5: Greenwashing Detection Matrix ---")
disclosure_threshold = df['Overall_Disclosure_Score_pct'].median()
cdp_threshold = 4  # CDP B rating = minimum for "coordinated climate action"
print(f"Thresholds: Disclosure median={disclosure_threshold:.1f}%, CDP={cdp_threshold} (B rating)")

def classify(row):
    disc, cdp = row['Overall_Disclosure_Score_pct'], row['CDP_Score_Encoded']
    if disc >= disclosure_threshold and cdp >= cdp_threshold: return 'Genuine Leaders'
    if disc >= disclosure_threshold and cdp < cdp_threshold: return 'Greenwashing Risk'
    if disc < disclosure_threshold and cdp >= cdp_threshold: return 'Underreporters'
    return 'Genuine Laggards'

df['Greenwashing_Category'] = df.apply(classify, axis=1)
print(df['Greenwashing_Category'].value_counts())

GW_COLORS = {'Genuine Leaders': '#1A6B3C', 'Greenwashing Risk': '#C0392B',
             'Underreporters': '#2980B9', 'Genuine Laggards': '#7F8C8D'}
fig, ax = plt.subplots(figsize=(13, 10))
for cat, color in GW_COLORS.items():
    mask = df['Greenwashing_Category'] == cat
    ax.scatter(df.loc[mask, 'Overall_Disclosure_Score_pct'], df.loc[mask, 'CDP_Score_Encoded'],
               c=color, label=f"{cat} (n={mask.sum()})", s=120, alpha=0.85, edgecolors='white', linewidth=0.8, zorder=3)
ax.axvline(x=disclosure_threshold, color='navy', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'Disclosure threshold (median={disclosure_threshold:.1f}%)')
ax.axhline(y=cdp_threshold, color='darkred', linestyle='--', linewidth=1.5, alpha=0.7,
           label=f'CDP threshold (B rating={cdp_threshold})')
for _, row in df.iterrows():
    ax.annotate(row['Company'].split(' ')[0][:10], (row['Overall_Disclosure_Score_pct'], row['CDP_Score_Encoded']),
                fontsize=6.5, ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', alpha=0.7)
ax.set_xlabel('Overall WikiRate Disclosure Score (%)'); ax.set_ylabel('CDP Climate Score (0=Not Disclosed -> 7=A)')
ax.set_title('Greenwashing Detection Matrix\nSelf-Reported Disclosure vs. Independently Verified CDP Performance',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(-2, 102); ax.set_ylim(-0.5, 7.8)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left'); ax.grid(True, alpha=0.2)
fig.text(0.5, -0.02, 'Note: Quadrant classification is based solely on quantitative disclosure and verification scores and\n'
                     'does not constitute a formal accusation of greenwashing, which would require legal determination.',
         ha='center', fontsize=8, style='italic', color='#555555')
plt.tight_layout(); plt.savefig(OUT + 'fig4_greenwashing_matrix.png'); plt.close()
print("  Saved: fig4_greenwashing_matrix.png")

## 7. HIERARCHICAL CLUSTERING (WARD LINKAGE)


In [ ]:
print("\n--- STEP 6: Hierarchical Clustering (Ward Linkage) ---")
Z = linkage(X_scaled, method='ward')
hc = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='ward')
hc_labels = hc.fit_predict(X_scaled)
hc_sil = silhouette_score(X_scaled, hc_labels)
ari = adjusted_rand_score(km_final.labels_, hc_labels)
print(f"Hierarchical Silhouette: {hc_sil:.4f} | ARI vs K-means: {ari:.4f}")

fig, ax = plt.subplots(figsize=(16, 7))
company_names = df['Company'].str.split(' ').str[0].tolist()
dendrogram(Z, labels=company_names, ax=ax, color_threshold=Z[-OPTIMAL_K+1, 2], leaf_rotation=90, leaf_font_size=7.5)
ax.set_title(f'Hierarchical Clustering Dendrogram (Ward Linkage) — K={OPTIMAL_K}\n'
             f'Silhouette: {hc_sil:.3f} | ARI vs K-means: {ari:.3f}', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Company'); ax.set_ylabel('Distance (Ward Linkage)')
ax.axhline(y=Z[-OPTIMAL_K+1, 2], color='red', linestyle='--', alpha=0.6, label=f'Cut threshold (K={OPTIMAL_K})')
ax.legend()
plt.tight_layout(); plt.savefig(OUT + 'fig5_dendrogram.png'); plt.close()
print("  Saved: fig5_dendrogram.png")

## 8. SUB-SEGMENT AND REGIONAL BREAKDOWN


In [ ]:
print("\n--- STEP 7: Sub-Segment and Regional Breakdown ---")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Cluster Composition by Sub-Segment and Region', fontsize=13, fontweight='bold')
seg_cross = pd.crosstab(df['Sub_segment'], df['Cluster_Label'])
seg_pct = seg_cross.div(seg_cross.sum(axis=1), axis=0) * 100
sns.heatmap(seg_pct, annot=True, fmt='.0f', cmap='Blues', ax=axes[0], linewidths=0.5,
            cbar_kws={'label': '% of sub-segment'})
axes[0].set_title('Sub-Segment Distribution (%)'); axes[0].set_xlabel(''); axes[0].set_ylabel('Sub-Segment')
plt.setp(axes[0].get_xticklabels(), rotation=25, ha='right', fontsize=9)

reg_cross = pd.crosstab(df['HQ_Region'], df['Cluster_Label'])
reg_pct = reg_cross.div(reg_cross.sum(axis=1), axis=0) * 100
sns.heatmap(reg_pct, annot=True, fmt='.0f', cmap='Greens', ax=axes[1], linewidths=0.5,
            cbar_kws={'label': '% of region'})
axes[1].set_title('Regional Distribution (%)'); axes[1].set_xlabel(''); axes[1].set_ylabel('HQ Region')
plt.setp(axes[1].get_xticklabels(), rotation=25, ha='right', fontsize=9)
plt.tight_layout(); plt.savefig(OUT + 'fig6_subsegment_region.png'); plt.close()
print("  Saved: fig6_subsegment_region.png")

## 9. SILHOUETTE PLOT


In [ ]:
print("\n--- STEP 8: Silhouette Plot ---")
sil_values = silhouette_samples(X_scaled, km_final.labels_)
fig, ax = plt.subplots(figsize=(10, 8))
y_lower = 10
for i in range(OPTIMAL_K):
    cluster_sil = np.sort(sil_values[km_final.labels_ == i])
    size = len(cluster_sil); y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil, alpha=0.7, color=CLUSTER_COLORS_K2[i])
    ax.text(-0.05, y_lower + 0.5 * size, f'C{i+1}', fontsize=10, color=CLUSTER_COLORS_K2[i], fontweight='bold')
    y_lower = y_upper + 10
ax.axvline(x=final_sil, color='red', linestyle='--', linewidth=1.5, label=f'Average: {final_sil:.3f}')
ax.set_title(f'Silhouette Plot — K={OPTIMAL_K} Clusters', fontsize=13, fontweight='bold')
ax.set_xlabel('Silhouette Coefficient'); ax.set_ylabel('Cluster'); ax.legend(); ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout(); plt.savefig(OUT + 'fig7_silhouette_plot.png'); plt.close()
print("  Saved: fig7_silhouette_plot.png")

# ══════════════════════════════════════════════════════════════════════════
# SUPPLEMENTARY ANALYSIS: K=3 AND K=4 (per tutor guidance)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SUPPLEMENTARY ANALYSIS: K=3 AND K=4")
print("=" * 60)

sil_scores_all = {2: final_sil}
for k in [3, 4]:
    km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
    km.fit(X_scaled)
    sil = silhouette_score(X_scaled, km.labels_)
    sil_scores_all[k] = sil
    means = df.groupby(pd.Series(km.labels_, name='c'))['Overall_Disclosure_Score_pct'].mean()
    rank = means.sort_values(ascending=False).index.tolist()
    label_map = {orig: i for i, orig in enumerate(rank)}
    df[f'K{k}'] = pd.Series(km.labels_).map(label_map).values
    print(f"\nK={k}: Silhouette={sil:.4f}")
    for c in range(k):
        mask = df[f'K{k}'] == c
        sub = df[mask]
        print(f"  Cluster {c+1} (n={mask.sum()}): Disclosure={sub['Overall_Disclosure_Score_pct'].mean():.1f}%, "
              f"CDP={sub['CDP_Score_Encoded'].mean():.2f}, NetZero={sub['Has_Net_Zero_Target'].mean()*100:.0f}%")

df['K2'] = df['Cluster'].map({c: i for i, c in enumerate(rank_order)})

## FIG 8: K=3 and K=4 PCA plots


In [ ]:
LABELS_K3 = ['Leaders (n=11)', 'Transitional (n=17)', 'Low Maturity (n=36)']
LABELS_K4 = ['Leaders (n=11)', 'Committed/Transitioning (n=12)', 'Emerging (n=21)', 'Low Maturity (n=20)']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for i in range(3):
    mask = df['K3'] == i
    axes[0].scatter(df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'], c=CLUSTER_COLORS_K3[i],
                    label=LABELS_K3[i], s=90, alpha=0.85, edgecolors='white', linewidth=0.8, zorder=3)
for _, row in df.iterrows():
    axes[0].annotate(row['Company'].split(' ')[0][:8], (row['PCA1'], row['PCA2']), fontsize=6,
                     ha='center', va='bottom', xytext=(0, 4), textcoords='offset points', alpha=0.7)
axes[0].set_xlabel(f'PC1 ({var_exp[0]:.1%})'); axes[0].set_ylabel(f'PC2 ({var_exp[1]:.1%})')
axes[0].set_title(f'K=3 Cluster Solution\n(Silhouette = {sil_scores_all[3]:.3f})', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10, loc='upper left'); axes[0].grid(True, alpha=0.25)
axes[0].text(0.02, 0.02, f'Silhouette = {sil_scores_all[3]:.3f} (below 0.3 threshold)', transform=axes[0].transAxes,
            fontsize=9, color='darkred', bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFE0E0', alpha=0.8))

for i in range(4):
    mask = df['K4'] == i
    axes[1].scatter(df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'], c=CLUSTER_COLORS_K4[i],
                    label=LABELS_K4[i], s=90, alpha=0.85, edgecolors='white', linewidth=0.8, zorder=3)
for _, row in df.iterrows():
    axes[1].annotate(row['Company'].split(' ')[0][:8], (row['PCA1'], row['PCA2']), fontsize=6,
                     ha='center', va='bottom', xytext=(0, 4), textcoords='offset points', alpha=0.7)
axes[1].set_xlabel(f'PC1 ({var_exp[0]:.1%})'); axes[1].set_ylabel(f'PC2 ({var_exp[1]:.1%})')
axes[1].set_title(f'K=4 Cluster Solution\n(Silhouette = {sil_scores_all[4]:.3f})', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9, loc='upper left'); axes[1].grid(True, alpha=0.25)
axes[1].text(0.02, 0.02, f'Silhouette = {sil_scores_all[4]:.3f} (below 0.3 threshold)', transform=axes[1].transAxes,
            fontsize=9, color='darkred', bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFE0E0', alpha=0.8))
plt.suptitle('Supplementary Analysis: K=3 and K=4 Cluster Solutions\n(shown for comparison; K=2 remains primary solution)',
            fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig(OUT + 'fig8_k3_k4_pca.png'); plt.close()
print("\n  Saved: fig8_k3_k4_pca.png")

## FIG 9: Comparison table (K=2/3/4)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6)); ax.axis('off')

def agg_table(kcol, k):
    d = df.groupby(kcol).agg(n=('Company', 'count'), disclosure=('Overall_Disclosure_Score_pct', 'mean'),
                              cdp=('CDP_Score_Encoded', 'mean'),
                              netzero=('Has_Net_Zero_Target', lambda x: f"{x.mean()*100:.0f}%")).round(1)
    return d

k2d, k3d, k4d = agg_table('K2', 2), agg_table('K3', 3), agg_table('K4', 4)

table_data = [
    ['K=2', f"{sil_scores_all[2]:.3f} \u2713", 'Leaders', str(k2d.loc[0,'n']), f"{k2d.loc[0,'disclosure']:.1f}%", f"{k2d.loc[0,'cdp']:.2f}", k2d.loc[0,'netzero']],
    ['', '', 'Developing', str(k2d.loc[1,'n']), f"{k2d.loc[1,'disclosure']:.1f}%", f"{k2d.loc[1,'cdp']:.2f}", k2d.loc[1,'netzero']],
    ['K=3', f"{sil_scores_all[3]:.3f} \u2717", 'Leaders', str(k3d.loc[0,'n']), f"{k3d.loc[0,'disclosure']:.1f}%", f"{k3d.loc[0,'cdp']:.2f}", k3d.loc[0,'netzero']],
    ['', '', 'Transitional', str(k3d.loc[1,'n']), f"{k3d.loc[1,'disclosure']:.1f}%", f"{k3d.loc[1,'cdp']:.2f}", k3d.loc[1,'netzero']],
    ['', '', 'Low Maturity', str(k3d.loc[2,'n']), f"{k3d.loc[2,'disclosure']:.1f}%", f"{k3d.loc[2,'cdp']:.2f}", k3d.loc[2,'netzero']],
    ['K=4', f"{sil_scores_all[4]:.3f} \u2717", 'Leaders', str(k4d.loc[0,'n']), f"{k4d.loc[0,'disclosure']:.1f}%", f"{k4d.loc[0,'cdp']:.2f}", k4d.loc[0,'netzero']],
    ['', '', 'Committed/Transitioning', str(k4d.loc[1,'n']), f"{k4d.loc[1,'disclosure']:.1f}%", f"{k4d.loc[1,'cdp']:.2f}", k4d.loc[1,'netzero']],
    ['', '', 'Emerging', str(k4d.loc[2,'n']), f"{k4d.loc[2,'disclosure']:.1f}%", f"{k4d.loc[2,'cdp']:.2f}", k4d.loc[2,'netzero']],
    ['', '', 'Low Maturity', str(k4d.loc[3,'n']), f"{k4d.loc[3,'disclosure']:.1f}%", f"{k4d.loc[3,'cdp']:.2f}", k4d.loc[3,'netzero']],
]
headers = ['K', 'Silhouette', 'Cluster Label', 'N', 'Disclosure (%)', 'CDP (0-7)', 'Net-Zero %']
tbl = ax.table(cellText=table_data, colLabels=headers, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 2.2)
for j in range(len(headers)):
    tbl[0, j].set_facecolor('#1A3C5E'); tbl[0, j].set_text_props(color='white', fontweight='bold')
row_colors = {0:'#EBF5FB',1:'#EBF5FB',2:'#FEF9E7',3:'#FEF9E7',4:'#FEF9E7',5:'#EAFAF1',6:'#EAFAF1',7:'#EAFAF1',8:'#EAFAF1'}
for row, color in row_colors.items():
    for j in range(len(headers)):
        tbl[row+1, j].set_facecolor(color)
ax.set_title('Comparison of K=2, K=3, and K=4 Cluster Solutions\n\u2713 = above 0.3 threshold | \u2717 = below 0.3 threshold',
            fontsize=13, fontweight='bold', pad=20)
plt.tight_layout(); plt.savefig(OUT + 'fig9_cluster_comparison_table.png'); plt.close()
print("  Saved: fig9_cluster_comparison_table.png")

## FIG 10: K=3 and K=4 radar charts


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw=dict(polar=True))
fig.suptitle('Supplementary Radar Charts: K=3 and K=4 Cluster Profiles\nvs Industry Average (dashed)',
            fontsize=13, fontweight='bold', y=1.01)
industry_avg = df[THEME_VARS].mean().values.tolist(); industry_avg += industry_avg[:1]

k3_labels = ['Leaders', 'Transitional', 'Low Maturity']
for i in range(3):
    mask = df['K3'] == i
    vals = df[mask][THEME_VARS].mean().values.tolist(); vals += vals[:1]
    ax = axes[0][i] if i < 2 else axes[1][0]
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(THEME_LABELS, size=9)
    ax.set_ylim(0, 100); ax.set_yticks([20, 40, 60, 80])
    ax.plot(angles, industry_avg, 'o--', color='grey', linewidth=1.5, alpha=0.5, label='Industry avg')
    ax.fill(angles, industry_avg, alpha=0.06, color='grey')
    ax.plot(angles, vals, 'o-', color=CLUSTER_COLORS_K3[i], linewidth=2.5)
    ax.fill(angles, vals, alpha=0.2, color=CLUSTER_COLORS_K3[i])
    ax.set_title(f'K=3: {k3_labels[i]} (n={mask.sum()})', size=10, fontweight='bold', pad=15)

k4_labels = ['Leaders', 'Committed', 'Emerging', 'Low Maturity']
ax = axes[1][1]
for i in range(4):
    mask = df['K4'] == i
    vals = df[mask][THEME_VARS].mean().values.tolist(); vals += vals[:1]
    ax.plot(angles, vals, 'o-', color=CLUSTER_COLORS_K4[i], linewidth=2, label=f'{k4_labels[i]} (n={mask.sum()})')
ax.set_xticks(angles[:-1]); ax.set_xticklabels(THEME_LABELS, size=9)
ax.set_ylim(0, 100); ax.set_yticks([20, 40, 60, 80])
ax.plot(angles, industry_avg, 'o--', color='grey', linewidth=1.5, alpha=0.5, label='Industry avg')
ax.set_title('K=4: All Clusters Overlaid', size=10, fontweight='bold', pad=15)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=8)
plt.tight_layout(); plt.savefig(OUT + 'fig10_k3_k4_radars.png'); plt.close()
print("  Saved: fig10_k3_k4_radars.png")

## SAVE FINAL DATASET


In [ ]:
df.to_csv(OUT + 'Fashion_Clustered_Dataset.csv', index=False)
print(f"\n✅ Final clustered dataset saved: {OUT}Fashion_Clustered_Dataset.csv")

## FINAL SUMMARY


In [ ]:
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"\nPrimary solution: K=2, Silhouette={final_sil:.4f}, ARI vs hierarchical={ari:.4f}")
print(f"Supplementary: K=3 Silhouette={sil_scores_all[3]:.4f} | K=4 Silhouette={sil_scores_all[4]:.4f}")
print("\nGreenwashing matrix:")
print(df['Greenwashing_Category'].value_counts())
print("\nAll 10 figures and final dataset saved to:", OUT)
print("Analysis complete.")